# AFML Review 1 — Small Transformer demo
Train one small model on a bounded FineWeb-Edu sample. Mamba training is a later milestone.

Run cells from top to bottom in Google Colab. A GPU is optional for this tiny run. The notebook has no stored results; the repository's `results/review1` folder contains the separately recorded pilot evidence.


In [ ]:
from pathlib import Path
import subprocess, sys, os
repo = Path('/content/afml-transformer-vs-state-space-models')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'review1/transformer-demo', '--single-branch',
                    'https://github.com/Gagan-1718/afml-transformer-vs-state-space-models.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


## 1. Check the environment
CPU works for a tiny run. If CUDA is available, the training script uses it automatically.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU runtime')
torch.set_num_threads(4)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## 2. Prepare the data
We split whole documents before tokenization and fit the tokenizer only on the training split. Data downloads can take time. Every run uses a fresh directory so earlier work is preserved.

In [ ]:
from datetime import datetime
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
data_dir = f'data/review1-{stamp}'
run_dir = f'runs/review1-{stamp}'
subprocess.run([sys.executable, 'scripts/review1_demo.py', 'prepare', '--data-dir', data_dir], check=True)


## 3. Inspect inputs and targets
The target is shifted one token ahead. Each row stays inside one source document.

In [ ]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(f'{data_dir}/tokenizer.json')
blocks = torch.load(f'{data_dir}/blocks.pt', weights_only=True)
row = blocks['train'][0]
print('Batch data:', blocks['train'].shape)
print('Input IDs: ', row[:12].tolist())
print('Target IDs:', row[1:13].tolist())
print('Input text:', tokenizer.decode(row[:-1].tolist()))


## 4. Train the Transformer
Two layers, width 128, four attention heads and a 128-token context. We run 300 updates. Validation chooses the saved checkpoint; this is not a final test score.

In [ ]:
subprocess.run([sys.executable, 'scripts/review1_demo.py', 'train',
                '--data-dir', data_dir, '--run-dir', run_dir], check=True)


## 5. Read the results
Falling loss means better next-token predictions. It does not by itself establish fluent generation or an advantage over Mamba.

In [ ]:
import json
from IPython.display import display, Image
summary = json.loads(Path(f'{run_dir}/summary.json').read_text())
print('Parameters:', summary['parameters'])
print('Initial validation loss:', summary['initial']['validation_loss'])
print('Best validation loss:', summary['best_validation_loss'])
print('Best validation perplexity:', summary['best_validation_perplexity'])
display(Image(filename=f'{run_dir}/loss_curve.png'))
print(Path(f'{run_dir}/samples.txt').read_text())


## 6. Try a prompt
This model continues text. It has not been instruction-tuned. Short or repetitive output is expected at this scale.

In [ ]:
subprocess.run([sys.executable, 'scripts/review1_demo.py', 'generate',
                '--run-dir', run_dir, '--prompt', 'The student learned that'], check=True)


## 7. Save the run before Colab disconnects
This downloads weights, tokenizer, results, and the data manifest. The archive does not include raw source documents.

In [ ]:
import shutil
from google.colab import files
shutil.copy2(f'{data_dir}/manifest.json', f'{run_dir}/manifest.json')
archive = shutil.make_archive(f'/content/review1-{stamp}', 'zip', run_dir)
files.download(archive)
